In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
import numpy as np
from peft import UIOrthoLoRAConfig, get_peft_model, TaskType, PeftConfig, PeftModel
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from tqdm import tqdm
from datasets import load_dataset
from torch.utils.data import DataLoader

In [2]:
# Count reference groups
with open("references/testset.txt") as f:
    num_refs = sum(1 for line in f if line.strip() == "") + 1

# Count system outputs
with open("references/system_outputs.txt") as f:
    num_sys = sum(1 for line in f)

print(f"Reference groups: {num_refs}, System outputs: {num_sys}")


Reference groups: 631, System outputs: 4693


In [3]:
original_ds = load_dataset("tuetschek/e2e_nlg")

In [4]:
# from datasets import load_dataset

# original_ds = load_dataset("tuetschek/e2e_nlg")

# # View a few examples
# for i in range(10):
#     record = original_ds["test"][i]
#     print("Meaning Representation (MR):", record["meaning_representation"])
#     print("Reference:", record.get("human_reference") or record.get("reference"))
#     print("-" * 60)


In [5]:
# for i, sentence in enumerate(original_ds["test"]["human_reference"]):
#     print(f"Sentence {i+1}: {sentence}")
#     print("-" * 80)

#     if i == 20:
#         break

In [6]:
def load_and_prepare(tokenizer):
    ds = load_dataset("tuetschek/e2e_nlg", trust_remote_code=True)

    def to_features(rec):
        prompt    = f"{rec['meaning_representation']} => "
        reference = rec.get("human_reference") or rec.get("reference", "")

        # 1️⃣  tokenize prompt **alone** (no padding, no special tokens)
        prompt_ids = tokenizer(prompt,
                            add_special_tokens=False,
                            padding=False,
                            truncation=False)["input_ids"]

        text       = prompt + reference + tokenizer.eos_token
        tok        = tokenizer(text,
                            truncation=True,
                            padding="max_length",
                            max_length=512)

        labels = tok["input_ids"].copy()
        labels[:len(prompt_ids)] = [-100] * len(prompt_ids)  # mask prompt
        labels = [l if l != tokenizer.pad_token_id else -100 for l in labels]  # mask padding

        tok["labels"]     = labels
        tok["prompt_ids"] = prompt_ids                        # ✅ real prompt
        return tok

    return ds.map(to_features, remove_columns=ds["train"].column_names)

In [7]:
# test_ds = original_ds["test"]

# Load tokenizer (use the same as your model)
gpt2_tokenizer = AutoTokenizer.from_pretrained("gpt2")  # or your model's tokenizer

# Detokenize model predictions
def decode_preds(preds):
    return gpt2_tokenizer.batch_decode(preds, skip_special_tokens=True)

# Print model generations vs gold labels
def print_preds_vs_gold(model_preds, labels):
    for i in range(min(10, len(model_preds))):
        pred = model_preds[i].strip()
        gold = labels[i].strip()
        print(f"💡 Prediction {i+1}: {pred}")
        print(f"✅ Gold Label {i+1}: {gold}")
        print("-" * 80)


/opt/anaconda3/envs/guyb_env2/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [8]:
INFERENCE_ARGS = {
    "num_beams": 10,
    "no_repeat_ngram_size": 4,
    "length_penalty": 0.9,
    "max_new_tokens": 64,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
def get_tokenizer(model_type):
    tokenizer = AutoTokenizer.from_pretrained(model_type)
    set_tokenizer(tokenizer, padding_side="right")
    return tokenizer

def set_contiguous(model):
    for m in model.modules():
        if hasattr(m, "parametrizations") and "weight" in m.parametrizations:
            base = m.parametrizations.weight[0].base
            if not base.is_contiguous():
                base.data = base.data.contiguous()

def set_tokenizer(tokenizer, padding_side):
    tokenizer.padding_side = padding_side
    if tokenizer.pad_token is None:
        tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
        print("Added new pad token:", tokenizer.pad_token)

def get_base_model(model_type, device, tokenizer):
    base_model = AutoModelForCausalLM.from_pretrained(model_type)
    base_model.resize_token_embeddings(len(tokenizer))
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model = base_model.to(device)    
    return base_model

def get_tokenizer_and_model(model_path: str, device):
    """
    Load a base model and inject a saved PEFT adapter from `model_path`.
    """
    # 1) Load the adapter config to get the original base model
    peft_config = PeftConfig.from_pretrained(model_path)
    base_model_name = peft_config.base_model_name_or_path

    # 2) Load base model and tokenizer
    tokenizer = get_tokenizer(base_model_name)

    base_model = get_base_model(base_model_name, device, tokenizer)

    # 3) Load the adapter into the base model
    model = PeftModel.from_pretrained(base_model, model_path)
    model = model.to(device)

    # 4) Ensure contiguous weights (optional)
    set_contiguous(model)

    return tokenizer, model, peft_config

In [10]:
tokenizer, model, peft_config = get_tokenizer_and_model("/home/guyb/gpufs/peft/notebooks/E2E/outputs/lora_models/lr_0.0002", device)

Added new pad token: <|pad|>


In [11]:
tokenizer.padding_side = "left"

In [12]:
ds = load_and_prepare(tokenizer)

Map:   0%|          | 0/4693 [00:00<?, ? examples/s]

In [13]:
example = ds["test"][0]

# Get the tokenized fields
input_ids   = example["input_ids"]
prompt_ids  = example["prompt_ids"]
labels      = example["labels"]
attention   = example["attention_mask"]

# Decode them
print("🔹 Prompt only:")
print(tokenizer.decode(prompt_ids, skip_special_tokens=False))

print("\n🔹 Full input (prompt + reference):")
print(tokenizer.decode(input_ids, skip_special_tokens=False))

print("\n🔹 Labels (non -100 only):")
label_ids = [i for i in labels if i != -100]
print(tokenizer.decode(label_ids, skip_special_tokens=False))


🔹 Prompt only:
name[Blue Spice], eatType[coffee shop], area[city centre] => 

🔹 Full input (prompt + reference):
<|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad|><|pad

In [ ]:
# len_prompt     = len(example["prompt_ids"])                # e.g. 13
# len_real_input = int(sum(example["attention_mask"]))       # prompt + reference
# len_target     = int(sum(np.array(example["labels"]) != -100))
# assert len_target == len_real_input - len_prompt


AssertionError: 

In [15]:
print(sum(list(map(lambda x: False if x==0 else True, ds["test"]["attention_mask"][0]))))
print(sum(list(map(lambda x: False if x==50256 else True, ds["test"]["input_ids"][0]))))
print(sum(list(map(lambda x: False if x==-100 else True, ds["test"]["labels"][0]))))

31
511
31


In [16]:
def collate_fn(batch):
        feats = [{"input_ids": b["prompt_ids"]} for b in batch]
        out = tokenizer.pad(
        feats,
        padding="longest",
        return_attention_mask=True,
        return_tensors="pt")
        return out      

dataloader = DataLoader(
    ds["test"].select(range(20)),
    batch_size=16,
    collate_fn=collate_fn,
)

In [17]:
gen_texts = []
for item in tqdm(dataloader):
    prompt_ids = item["input_ids"].to(device)
    attention_mask = item["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=prompt_ids,
            attention_mask=attention_mask,
            max_new_tokens=INFERENCE_ARGS["max_new_tokens"],
            num_beams=INFERENCE_ARGS["num_beams"],
            no_repeat_ngram_size=INFERENCE_ARGS["no_repeat_ngram_size"],
            length_penalty=INFERENCE_ARGS["length_penalty"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # strip the prompt text (“=>” part) to isolate hypothesis
    preds = [p.split("=>")[-1].strip() for p in preds]
    gen_texts.extend(preds)

100%|██████████| 2/2 [00:02<00:00,  1.40s/it]


In [18]:
subset = original_ds["test"].select(range(20))

for i, example in enumerate(subset):
    print(f"Sample {i+1}")
    print("💬 Meaning Representation:", example["meaning_representation"])
    print("🎯 Human Reference:", example["human_reference"])
    print("📝 Model Prediction:", gen_texts[i])
    print("-" * 80)


Sample 1
💬 Meaning Representation: name[Blue Spice], eatType[coffee shop], area[city centre]
🎯 Human Reference: A coffee shop in the city centre area called Blue Spice.
📝 Model Prediction: Blue Spice is a coffee shop located in the city centre.
--------------------------------------------------------------------------------
Sample 2
💬 Meaning Representation: name[Blue Spice], eatType[coffee shop], area[city centre]
🎯 Human Reference: Blue Spice is a coffee shop in city centre.
📝 Model Prediction: Blue Spice is a coffee shop located in the city centre.
--------------------------------------------------------------------------------
Sample 3
💬 Meaning Representation: name[Blue Spice], eatType[coffee shop], area[riverside]
🎯 Human Reference: There is a coffee shop Blue Spice in the riverside area.
📝 Model Prediction: Blue Spice is a coffee shop in the riverside area.
--------------------------------------------------------------------------------
Sample 4
💬 Meaning Representation: name[Bl